In [ ]:
import os
import sympy
import pickle
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
def find_upper_threshold(values):
    """
    Find the upper threshold where 90% of the data lies below this threshold.

    Args:
        values (list of float): A list of values ranging from 0.00 to 100.00.

    Returns:
        float: The threshold value that includes 90% of the data.
    """
    if not values:
        raise ValueError("The input list is empty.")

    # Sort the values in ascending order
    sorted_values = sorted(values)

    # Calculate the index corresponding to 95% of the data
    threshold_index = int(len(sorted_values) * 0.95) - 1

    # Handle edge case when threshold_index is out of bounds
    threshold_index = max(0, min(threshold_index, len(sorted_values) - 1))

    return sorted_values[threshold_index]

In [ ]:
def leave_second_element(List):
    lst=[]

    for x in List:
#        if x[1]>0.0 and x[1]<100.0:
#            lst.append(x[2])
        lst.append(x[1])
    #print(type(unique_arr))
#     for sublist in lst:
#         result_list.append(list(sublist[0:2]))

    return lst


In [ ]:
# from skimage import color

def extractColors(base_dir, search_term=""):
    filename = search_term + ".png"

    resize_image_dir = base_dir

    img_url = resize_image_dir + '/' + filename

    # Open the image file
    image = Image.open(img_url)

    # Get the size of the image in pixels
    width, height = image.size

    # Create an empty list to store the RGB values for each pixel
    rgb_list = []
    lab_list=[]
    # Loop over each pixel in the image and extract the RGB values
    for y in range(height):
        for x in range(width):
            pixel = image.getpixel((x, y))
            if isinstance(pixel, int):
                r, g, b = pixel, pixel, pixel
            else:
                r, g, b = pixel[:3]
            rgb_list.append((r, g, b))

    for (r, g, b) in rgb_list:
        if isinstance(r, int) and isinstance(g, int) and isinstance(b, int) and 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
            # Convert RGB to XYZ
            r_norm = r / 255
            g_norm = g / 255
            b_norm = b / 255

            r_lin = r_norm if r_norm <= 0.04045 else ((r_norm + 0.055) / 1.055) ** 2.4
            g_lin = g_norm if g_norm <= 0.04045 else ((g_norm + 0.055) / 1.055) ** 2.4
            b_lin = b_norm if b_norm <= 0.04045 else ((b_norm + 0.055) / 1.055) ** 2.4

            x = r_lin * 0.4124564 + g_lin * 0.3575761 + b_lin * 0.1804375
            y = r_lin * 0.2126729 + g_lin * 0.7151522 + b_lin * 0.0721750
            z = r_lin * 0.0193339 + g_lin * 0.1191920 + b_lin * 0.9503041

            # Convert XYZ to Lab
            xn = 0.95047
            yn = 1.00000
            zn = 1.08883

            f_x = x ** (1/3) if x > 0.008856 else 7.787 * x + 16 / 116
            f_y = y ** (1/3) if y > 0.008856 else 7.787 * y + 16 / 116
            f_z = z ** (1/3) if z > 0.008856 else 7.787 * z + 16 / 116

            L = 116 * f_y - 16
            a = 500 * (f_x - f_y)
            b = 200 * (f_y - f_z)

            lab_list.append((L, a, b))
        else:
            print("Invalid RGB value:", (r, g, b))

    if not lab_list:
        return None

    # Convert CIELAB values to CIELCH values
    lch_list = []

    for lab in lab_list:
        L, a, b = lab
        C = np.sqrt(a**2 + b**2)
        H = np.degrees(np.arctan2(b, a))

        lch_list.append((L, C, H))

    # Create Dataframe
    df = pd.DataFrame({'cielab': lab_list, 'cielch': lch_list})
    df['occurrence'] = df.groupby('cielab')['cielab'].transform('count')
    df['ratios'] = df["occurrence"]/df["occurrence"].sum()



    return df["cielch"]

In [ ]:
import os

# Directory where resized 15 achromatic term images are saved
base_dir = './Images/resize_images'

# List of target achromatic terms
achromatic_words = [
    "black", "pitchblack", "pitchdark", "jetblack", "blackish",
    "grey", "silver", "slategray", "smokegray", "silvery",
    "white", "snowywhite", "milkwhite", "milkywhite", "chalkwhite"
]

# Initialize result list
upper_threshold_list = []

# Filter files: only those matching achromatic_words and existing in base_dir
files = [f"{word}.png" for word in achromatic_words if os.path.exists(os.path.join(base_dir, f"{word}.png"))]
print(f"Found {len(files)} matching achromatic images.")

# Process each file
for file_name in files:
    search_term = os.path.splitext(file_name)[0]

    try:
        df_cielch = extractColors(base_dir, search_term)
        if df_cielch is None:
            print(f"No data extracted for {file_name}.")
            continue
    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        continue

    # Extract the second elements (e.g., Chroma or Lightness)
    second_elements = leave_second_element(df_cielch)

    # Calculate threshold
    upper_threshold = find_upper_threshold(second_elements)

    # Save result
    upper_threshold_list.append(upper_threshold)
    print(f"{file_name} → Upper threshold: {upper_threshold}")

print("Processing complete!")


In [ ]:
# Define color mapping for search terms
color_mapping = {
    "blackish": "black", "pitchblack": "black", "black": "black",
    "pitchdark": "black", "jetblack": "black",
    "white": "lightgrey", "chalkwhite": "lightgrey", "snowywhite": "lightgrey",
    "milkywhite": "lightgrey", "milkwhite": "lightgrey",
    "silvery": "darkgrey", "slategray": "darkgrey", "smokegray": "darkgrey",
    "silver": "darkgrey", "grey": "darkgrey"
}

def plot_individual_histograms_with_titles(base_dir, output_folder):
    """
    Plot individual histograms for search terms with global max threshold,
    using a compact layout with a partial broken axis (50-90 omitted),
    and display color names as titles above each graph.

    Args:
        base_dir (str): The directory containing the .png files.
        output_folder (str): The folder to save individual histogram images.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    custom_order = [
        "white", "chalkwhite", "snowywhite", "milkywhite", "milkwhite",
        "grey", "silvery", "slategray", "smokegray", "silver",
        "black", "blackish", "pitchblack", "pitchdark", "jetblack"
    ]

    files = [f"{word}.png" for word in achromatic_words if os.path.exists(os.path.join(base_dir, f"{word}.png"))]

    sorted_files = sorted(files, key=lambda x: custom_order.index(os.path.splitext(x)[0]) if os.path.splitext(x)[0] in custom_order else len(custom_order))

    thresholds = {}
    max_threshold = None

    for file_name in sorted_files:
        search_term = os.path.splitext(file_name)[0]
        try:
            df_cielch = extractColors(base_dir, search_term)
            if df_cielch is None:
                print(f"No data extracted for {file_name}.")
                continue

            second_elements = leave_second_element(df_cielch)
            threshold = find_upper_threshold(second_elements)
            thresholds[search_term] = threshold
            if max_threshold is None or threshold > max_threshold:
                max_threshold = threshold
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    for file_name in sorted_files:
        search_term = os.path.splitext(file_name)[0]
        bar_color = color_mapping.get(search_term, "blue")

        try:
            df_cielch = extractColors(base_dir, search_term)
            if df_cielch is None:
                print(f"No data extracted for {file_name}.")
                continue

            second_elements = leave_second_element(df_cielch)
            threshold = thresholds.get(search_term, None)

            # Create compact broken axis plot
            fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, gridspec_kw={'width_ratios': [5, 1]}, figsize=(4, 3))

            # Plot left part of histogram (0 to 50)
            ax1.hist(second_elements, bins=50, range=(0, 50), color=bar_color, alpha=0.7, edgecolor='black')
            if threshold is not None and threshold <= 50:
                # if threshold != max_threshold:
                ax1.axvline(x=threshold, color='red', linestyle='--', label="Threshold")
                ax1.text(threshold + 1, ax1.get_ylim()[1] * 0.9, f"{threshold:.2f}", color='red', fontsize=8)
            if max_threshold is not None and max_threshold <= 50:
                ax1.axvline(x=max_threshold, color='green', linestyle='-', label="Max Threshold")
                ax1.text(max_threshold + 1, ax1.get_ylim()[1] * 0.8, f"{max_threshold:.2f}", color='green', fontsize=8)
            ax1.set_xlim(0, 50)
            ax1.set_xticks(range(0, 51, 10))
            ax1.set_xlabel("")
            ax1.set_ylabel("Frequency", fontsize=8)
            ax1.legend(fontsize=6)

            # Plot right part of histogram (90 to 100)
            ax2.hist(second_elements, bins=10, range=(90, 100), color=bar_color, alpha=0.7, edgecolor='black')
            if threshold is not None and 90 <= threshold <= 100:
                ax2.axvline(x=threshold, color='red', linestyle='--', label="Threshold")
            if max_threshold is not None and 90 <= max_threshold <= 100:
                ax2.axvline(x=max_threshold, color='green', linestyle='-', label="Max Threshold")
            ax2.set_xlim(90, 100)
            ax2.set_xticks([90, 100])
            ax2.set_xlabel("")

            # Add break marks
            d = 0.015
            kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
            ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)
            ax1.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

            kwargs.update(transform=ax2.transAxes)
            ax2.plot((-d, +d), (-d, +d), **kwargs)
            ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)

            # Add common x-axis label
            fig.text(0.56, 0.02, "Chroma (0-100)", ha='center', fontsize=10)

            # Add color name as title
            fig.suptitle(search_term.capitalize(), fontsize=12, y=0.92 ,x=0.56)

            # Save histogram with titles
            plt.tight_layout(rect=[0, 0.05, 1, 0.99])
            save_path = os.path.join(output_folder, f"{search_term}_histogram_with_title.png")
            plt.savefig(save_path, dpi=300)
            print(f"Saved histogram for {search_term} with title to {save_path}")
            plt.close()

        except Exception as e:
            print(f"Error processing {file_name}: {e}")


In [ ]:
# Example usage
output_folder = './Visualizations/Threshold of Chroma' #folder that stores each achromatic graph
plot_individual_histograms_with_titles(base_dir, output_folder)